In [ ]:
import requests
import pandas as pd
import time
# Function to call NamSor API and get likelyGender
def get_gender(first_name, api_key="c27611bc5cc8dec7db8b744dce2761f5"):
    url = f"https://v2.namsor.com/NamSorAPIv2/api2/json/gender/{first_name}/"
    headers = {
        "X-API-KEY": api_key,
        "Accept": "application/json"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        data = response.json()
        return data.get("likelyGender", "unknown")  # Return 'unknown' if gender not found
    else:
        return "error"
# Function to process dataset in batches of 200
def process_gender_in_batches(df, batch_size=200):
    df['Gender_API'] = ''  # Initialize the Gender_API column
    for start in range(0, len(df), batch_size):
        end = start + batch_size
        batch = df[start:end]
        print(f"Processing batch {start+1} to {end}...")
        for idx, row in batch.iterrows():
            # Check if fullname is a string
            if isinstance(row['fullname'], str):
                first_name = row['fullname'].strip().split()[0]  # Only take the first name
                gender = get_gender(first_name)
                df.at[idx, 'Gender_API'] = gender
            else:
                df.at[idx, 'Gender_API'] = 'invalid'  # Mark invalid rows
            time.sleep(0.1)  # To avoid overloading the API (rate-limiting)
    return df
# Example usage
if __name__ == "__main__":
    # Load your dataset
    df = pd.read_excel('/content/updated 4th-AUG.xlsx')
    # Process dataset and add 'Gender_API' column
    df = process_gender_in_batches(df)
    # Save the updated dataset
    df.to_excel('/content/4th-AUG.xlsx_unkowns.xlsx', index=False)
    #print("Gender recognition completed and saved to '/content/July_gender_recognizer_1st_5000.xlsx'.")